<a href="https://colab.research.google.com/github/Abhi712-ui/Austin-Zoning-RAG-Project/blob/main/municode_exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pathlib import Path
import json
import requests

API = "https://api.municode.com"
STATE = "TX"
CLIENT = "Austin"
CLIENT_ID = "1113"
PRODUCT_ID = 15303
JOB_ID = 488379
TITLE_25_ID = "TIT25LADE"
ZONING_SECTION_ID = "TIT25LADE_CH25-2ZO"

In [2]:
#method for making api requests
def get_api(path, params=None):
    response = requests.get(
        f"{API}/{path}",
        params=params
    )
    response.raise_for_status()
    return response.json()

In [3]:
def find_nodes(obj, search_text):
    matches = []
    if isinstance(obj, dict):
        node_text = "".join(
            str(value) for value in obj.values()
            if isinstance(value, (str, int, float))
        )

        if search_text.lower() in node_text.lower():
            matches.append(obj)

        for value in obj.values():
            matches.extend(find_nodes(value, search_text))

    elif isinstance(obj, list):
        for item in obj: matches.extend(find_nodes(item, search_text))

    return matches

In [4]:
path = f"/Clients/{CLIENT_ID}"
client = get_api(path)
print(json.dumps(client, indent=4))

{
    "ClientID": 1113,
    "ClientCode": null,
    "ClientName": "Austin",
    "State": {
        "StateID": 43,
        "StateName": "Texas",
        "StateAbbreviation": "TX"
    },
    "Address": "301 West Second St 2nd Floor Suite 2030",
    "City": "Austin",
    "ZipCode": "78701",
    "Website": "www.austintexas.gov",
    "MeetingsUrl": null,
    "CivicGovUrl": null,
    "SocialMediaArchiveUrl": "https://austintexas.gov.archivesocial.com/",
    "SeeClickFixIntegrationUrl": null,
    "NextRequestUrl": null,
    "TenantCode": null,
    "LegacyTenantCode": null
}


In [5]:
path = f"/ClientContent/{CLIENT_ID}"
products = get_api(path)
print(type(products))
print(json.dumps(products, indent=4))

<class 'dict'>
{
    "codes": [
        {
            "productName": "Code of Ordinances",
            "productId": 15302,
            "latestUpdatedDate": "2026-08-21T16:35:58",
            "hasOrdbank": true,
            "contentTypeId": "CODES",
            "newOrdCount": 3,
            "hideInLibrary": false,
            "hasPdfDownloadEnabled": false,
            "hasPdf": false,
            "publicationId": 178,
            "showLastModifiedPublicationDate": true
        },
        {
            "productName": "Land Development Code",
            "productId": 15303,
            "latestUpdatedDate": "2026-08-21T17:03:12",
            "hasOrdbank": true,
            "contentTypeId": "CODES",
            "newOrdCount": 0,
            "hideInLibrary": false,
            "hasPdfDownloadEnabled": false,
            "hasPdf": false,
            "publicationId": 179,
            "showLastModifiedPublicationDate": true
        },
        {
            "productName": "Building Criteria Man

In [6]:
land_development = next(
    product for product in products["codes"]
    if product["productName"].lower() == "land development code"
)

product_id = land_development["productId"]
print(json.dumps(product_id, indent=4))
print(f"Product Id: {product_id}")

15303
Product Id: 15303


In [7]:
path = f"/Jobs/latest/{PRODUCT_ID}"
current_version = get_api(path)
print(json.dumps(current_version, indent=4))

{
    "Id": 497334,
    "Name": "Supplement 174",
    "ProductId": 15303,
    "Product": null,
    "PublishDate": "2026-05-28T04:00:00",
    "MaxTrackingDate": "2026-08-21T17:03:12",
    "OnlinePostDate": "2026-08-24T20:03:55.53031",
    "IsPublished": false,
    "IsLatest": true,
    "BannerText": "THE CODE\r\nOF THE CITY OF\r\nAUSTIN, TEXAS\r\n\r\nCodified through\r\nOrdinance No. 20260423-048, effective September 1, 2026.\r\n(Supp. No. 174)",
    "PublicationContentOrigin": "FullService_G2H",
    "OnlineDate": "2026-08-21T17:03:12"
}


In [8]:
path = f"/codesToc"
params = {
    "jobId": JOB_ID,
    "productId": PRODUCT_ID
}
get_toc = get_api(path, params)
print(json.dumps(get_toc, indent=4))

{
    "Id": "15303",
    "Heading": "Land Development Code",
    "NodeDepth": -1,
    "HasChildren": true,
    "ParentId": null,
    "DocOrderId": -1,
    "Children": [
        {
            "Id": "THCOAUTE",
            "Heading": "THE CODE OF THE CITY OF AUSTIN, TEXAS ",
            "NodeDepth": 1,
            "HasChildren": true,
            "ParentId": "15303",
            "DocOrderId": 1,
            "Children": [],
            "Data": {
                "NodeKey": null,
                "IsUpdated": false,
                "IsAmended": false,
                "HasAmendedDescendant": false,
                "CompareStatus": 4,
                "DocType": 1,
                "DepthOverride": null,
                "ChunkGroupStartingId": null
            }
        },
        {
            "Id": "SUHITA",
            "Heading": "SUPPLEMENT HISTORY TABLE",
            "NodeDepth": 1,
            "HasChildren": false,
            "ParentId": "15303",
            "DocOrderId": 4,
            "

In [9]:
path = f"/codesToc/children"
params = {
    "jobId": JOB_ID,
    "nodeId": TITLE_25_ID,
    "productId": PRODUCT_ID
}
title_25_children = get_api(path, params)
print(json.dumps(title_25_children, indent=4))

[
    {
        "Id": "TIT25LADE_CH25-1GEREPR",
        "Heading": "CHAPTER 25-1. - GENERAL REQUIREMENTS AND PROCEDURES.",
        "NodeDepth": 2,
        "HasChildren": true,
        "ParentId": "TIT25LADE",
        "DocOrderId": 6,
        "Children": [],
        "Data": {
            "NodeKey": null,
            "IsUpdated": false,
            "IsAmended": false,
            "HasAmendedDescendant": false,
            "CompareStatus": 4,
            "DocType": 1,
            "DepthOverride": null,
            "ChunkGroupStartingId": null
        }
    },
    {
        "Id": "TIT25LADE_CH25-2ZO",
        "Heading": "CHAPTER 25-2. - ZONING.",
        "NodeDepth": 2,
        "HasChildren": true,
        "ParentId": "TIT25LADE",
        "DocOrderId": 222,
        "Children": [],
        "Data": {
            "NodeKey": null,
            "IsUpdated": false,
            "IsAmended": false,
            "HasAmendedDescendant": false,
            "CompareStatus": 4,
            "DocType": 1,


In [10]:
path = f"/codesToc/children"
params = {
    "jobId": JOB_ID,
    "nodeId": ZONING_SECTION_ID,
    "productId": PRODUCT_ID
}
zoning_children = get_api(path, params)
print(json.dumps(zoning_children, indent=4))

[
    {
        "Id": "TIT25LADE_CH25-2ZO_SUBCHAPTER_AZOUSDIMADIDE",
        "Heading": "SUBCHAPTER A. - ZONING USES, DISTRICTS, AND MAP; DISTRICT DESIGNATIONS.",
        "NodeDepth": 3,
        "HasChildren": true,
        "ParentId": "TIT25LADE_CH25-2ZO",
        "DocOrderId": 223,
        "Children": [],
        "Data": {
            "NodeKey": null,
            "IsUpdated": false,
            "IsAmended": false,
            "HasAmendedDescendant": false,
            "CompareStatus": 4,
            "DocType": 1,
            "DepthOverride": null,
            "ChunkGroupStartingId": null
        }
    },
    {
        "Id": "TIT25LADE_CH25-2ZO_SUBCHAPTER_BZOPRSPRECEDI",
        "Heading": "SUBCHAPTER B. - ZONING PROCEDURES; SPECIAL REQUIREMENTS FOR CERTAIN DISTRICTS.",
        "NodeDepth": 3,
        "HasChildren": true,
        "ParentId": "TIT25LADE_CH25-2ZO",
        "DocOrderId": 315,
        "Children": [],
        "Data": {
            "NodeKey": null,
            "IsUpdated": 

In [11]:
def get_children(node_id):
    path = f"/codesToc/children"
    params = {
        "jobId": JOB_ID,
        "nodeId": node_id,
        "productId": PRODUCT_ID
    }
    return get_api(path, params)

In [12]:
def walk_tree(node, hierarchy=None):
    if hierarchy is None:
        hierarchy = []

    current_hierarchy = hierarchy + [node["Heading"]]

    nodes = [{
        "id": node["Id"],
        "heading": node["Heading"],
        "node_depth": node["NodeDepth"],
        "parent_id": node["ParentId"],
        "has_children": node["HasChildren"],
        "doc_order_id": node["DocOrderId"],
        "hierarchy": current_hierarchy
    }]

    if node["HasChildren"]:
        children = get_children(node["Id"])
        for child in children:
            nodes.extend(
                walk_tree(
                    child,
                    current_hierarchy
                )
            )
    return nodes

In [13]:
all_zoning_nodes = []
for child in zoning_children:
    all_zoning_nodes.extend(
        walk_tree(child, hierarchy=["CHAPTER 25-2. ZONING"])
    )

print("Total zoning nodes:", len(all_zoning_nodes))
print(json.dumps(all_zoning_nodes, indent=4))

Total zoning nodes: 664
[
    {
        "id": "TIT25LADE_CH25-2ZO_SUBCHAPTER_AZOUSDIMADIDE",
        "heading": "SUBCHAPTER A. - ZONING USES, DISTRICTS, AND MAP; DISTRICT DESIGNATIONS.",
        "node_depth": 3,
        "parent_id": "TIT25LADE_CH25-2ZO",
        "has_children": true,
        "doc_order_id": 223,
        "hierarchy": [
            "CHAPTER 25-2. ZONING",
            "SUBCHAPTER A. - ZONING USES, DISTRICTS, AND MAP; DISTRICT DESIGNATIONS."
        ]
    },
    {
        "id": "TIT25LADE_CH25-2ZO_SUBCHAPTER_AZOUSDIMADIDE_ART1ZOUS",
        "heading": "ARTICLE 1. - ZONING USES.",
        "node_depth": 4,
        "parent_id": "TIT25LADE_CH25-2ZO_SUBCHAPTER_AZOUSDIMADIDE",
        "has_children": true,
        "doc_order_id": 224,
        "hierarchy": [
            "CHAPTER 25-2. ZONING",
            "SUBCHAPTER A. - ZONING USES, DISTRICTS, AND MAP; DISTRICT DESIGNATIONS.",
            "ARTICLE 1. - ZONING USES."
        ]
    },
    {
        "id": "TIT25LADE_CH25-2ZO_SUBCH

In [14]:
leaf_nodes = [
    node for node in all_zoning_nodes
    if not node["has_children"]
]

print(json.dumps(leaf_nodes[0], indent=4))


{
    "id": "TIT25LADE_CH25-2ZO_SUBCHAPTER_AZOUSDIMADIDE_ART1ZOUS_S25-2-1USCL",
    "heading": "\u00a7 25-2-1 - USE CLASSIFICATIONS.",
    "node_depth": 5,
    "parent_id": "TIT25LADE_CH25-2ZO_SUBCHAPTER_AZOUSDIMADIDE_ART1ZOUS",
    "has_children": false,
    "doc_order_id": 225,
    "hierarchy": [
        "CHAPTER 25-2. ZONING",
        "SUBCHAPTER A. - ZONING USES, DISTRICTS, AND MAP; DISTRICT DESIGNATIONS.",
        "ARTICLE 1. - ZONING USES.",
        "\u00a7 25-2-1 - USE CLASSIFICATIONS."
    ]
}


In [15]:
def get_content(node_id):
    print(f"Fetching children of: {node_id}")
    path = f"/CodesContent"
    params={
            "jobId": JOB_ID,
            "nodeId": node_id,
            "productId": PRODUCT_ID
        }
    return get_api(path, params)

In [16]:
test_content = get_content(
    leaf_nodes[0]["id"]
)
print(json.dumps(test_content, indent=4))

Fetching children of: TIT25LADE_CH25-2ZO_SUBCHAPTER_AZOUSDIMADIDE_ART1ZOUS_S25-2-1USCL
{
    "Docs": [
        {
            "DocType": 1,
            "IsAmended": false,
            "IsUpdated": false,
            "CompareStatus": 4,
            "DocOrderId": 223,
            "AmendedBy": [],
            "Notes": [],
            "Drafts": [],
            "ChunkGroupStartingNodeId": "TIT25LADE_CH25-2ZO_SUBCHAPTER_AZOUSDIMADIDE",
            "NodeDepth": 3,
            "TitleHtml": "<div class=\"chunk-title\">SUBCHAPTER A. - ZONING USES, DISTRICTS, AND MAP; DISTRICT DESIGNATIONS.</div>",
            "ShouldShowMiniToc": false,
            "Id": "TIT25LADE_CH25-2ZO_SUBCHAPTER_AZOUSDIMADIDE",
            "Title": "SUBCHAPTER A. - ZONING USES, DISTRICTS, AND MAP; DISTRICT DESIGNATIONS.",
            "Content": "<div class=\"chunk-content\"></div>",
            "SortDate": null,
            "Footnotes": null
        },
        {
            "DocType": 1,
            "IsAmended": false,
    

In [17]:
docs = test_content["Docs"]
i = 0
for doc in docs:
    i += 1
    print(f"{i}. {doc["Id"]} -> {doc["Title"]}")

1. TIT25LADE_CH25-2ZO_SUBCHAPTER_AZOUSDIMADIDE -> SUBCHAPTER A. - ZONING USES, DISTRICTS, AND MAP; DISTRICT DESIGNATIONS.
2. TIT25LADE_CH25-2ZO_SUBCHAPTER_AZOUSDIMADIDE_ART1ZOUS -> ARTICLE 1. - ZONING USES.
3. TIT25LADE_CH25-2ZO_SUBCHAPTER_AZOUSDIMADIDE_ART1ZOUS_S25-2-1USCL -> § 25-2-1 - USE CLASSIFICATIONS.
4. TIT25LADE_CH25-2ZO_SUBCHAPTER_AZOUSDIMADIDE_ART1ZOUS_S25-2-2DEUSCL -> § 25-2-2 - DETERMINATION OF USE CLASSIFICATION.
5. TIT25LADE_CH25-2ZO_SUBCHAPTER_AZOUSDIMADIDE_ART1ZOUS_S25-2-3REUSDE -> § 25-2-3 - RESIDENTIAL USES DESCRIBED.
6. TIT25LADE_CH25-2ZO_SUBCHAPTER_AZOUSDIMADIDE_ART1ZOUS_S25-2-4COUSDE -> § 25-2-4 - COMMERCIAL USES DESCRIBED.
7. TIT25LADE_CH25-2ZO_SUBCHAPTER_AZOUSDIMADIDE_ART1ZOUS_S25-2-5INUSDE -> § 25-2-5 - INDUSTRIAL USES DESCRIBED.
8. TIT25LADE_CH25-2ZO_SUBCHAPTER_AZOUSDIMADIDE_ART1ZOUS_S25-2-6CIUSDE -> § 25-2-6 - CIVIC USES DESCRIBED.
9. TIT25LADE_CH25-2ZO_SUBCHAPTER_AZOUSDIMADIDE_ART1ZOUS_S25-2-7AGUSDE -> § 25-2-7 - AGRICULTURAL USES DESCRIBED.
10. TIT25LADE_CH

In [18]:
from bs4 import BeautifulSoup
import os
from google.colab import drive

drive.mount("/content/drive")
corpus_dir = "/content/drive/MyDrive/corpus"
os.makedirs(corpus_dir, exist_ok=True)

heading = docs[4]["Title"]
soup = BeautifulSoup(docs[4]["Content"],'html.parser')
text = soup.get_text("\n", strip=True)
file_path = os.path.join(corpus_dir, f"{heading}.txt")

with open(file_path, "w", encoding="utf-8") as f:
    f.write(text)

print(f"Wrote Text Successfully at {file_path}")

Mounted at /content/drive
Wrote Text Successfully at /content/drive/MyDrive/corpus/§ 25-2-3 - RESIDENTIAL USES DESCRIBED..txt


In [19]:
def save_section(section):
    corpus_dir = "/content/drive/MyDrive/corpus"
    heading = section["Title"]
    soup = BeautifulSoup(section["Content"], "html.parser")
    text = soup.get_text("\n", strip=True)
    if not text:
        print(f"Skipping empty doc {heading}")
        return
    file_path = os.path.join(corpus_dir, f"{heading}.txt")
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(text)
    print(f"Wrote Text Successfully at {file_path}")

In [20]:
def parse_section(section):
    heading = section["Title"]
    soup = BeautifulSoup(section["Content"], "html.parser")
    text = soup.get_text("\n", strip=True)
    return {
        "id": section["Id"],
        "heading": heading,
        "text": text
    }

In [21]:
print(parse_section(docs[5])["text"])

(A)
Commercial uses include the sale, rental, servicing, and distribution of goods, and
               the provision of services, other than those classified as industrial or civic uses.
(B)
Commercial use classifications are described as follows:
(1)
ADMINISTRATIVE AND BUSINESS OFFICES use is the use of a site for the provision of
               executive, management, or administrative services. This use includes:
(a)
administrative offices and services, including real estate, insurance, property management,
               investment, personnel, travel, secretarial, telephone answering, and photocopy and
               reproduction; and
(b)
business offices for public utilities, organizations, associations, and other use
               classifications if the service rendered is customarily associated with administrative
               office services.
(2)
AGRICULTURAL SALES AND SERVICES use is the use of a site for the on-site sale of feed,
               grain, fertilizers, pesticide

In [22]:
print(parse_section(docs[6])["text"])

(A)
Industrial uses include the on-site extraction or production of goods by non-agricultural
               methods, and the storage and distribution of products.
(B)
Industrial use classifications are described as follows:
(1)
BASIC INDUSTRY use is the use of a site for:
(a)
the basic processing and manufacturing of materials or products predominately from
               extracted or raw materials;
(b)
storage or manufacturing processes that involve flammable or explosive materials;
               or
(c)
storage or manufacturing processes that involve hazardous or commonly recognized offensive
               conditions, including poultry processing.
(2)
CUSTOM MANUFACTURING use is the use of a site for on-site production of goods by the
               use of hand tools, domestic mechanical equipment not exceeding five horsepower, or
               a single kiln not exceeding 12 kilowatts, and the incidental sale of those goods.
               This use includes candle-making shops and

In [23]:
for doc in docs[:10]:
    save_section(doc)

Skipping empty doc SUBCHAPTER A. - ZONING USES, DISTRICTS, AND MAP; DISTRICT DESIGNATIONS.
Skipping empty doc ARTICLE 1. - ZONING USES.
Wrote Text Successfully at /content/drive/MyDrive/corpus/§ 25-2-1 - USE CLASSIFICATIONS..txt
Wrote Text Successfully at /content/drive/MyDrive/corpus/§ 25-2-2 - DETERMINATION OF USE CLASSIFICATION..txt
Wrote Text Successfully at /content/drive/MyDrive/corpus/§ 25-2-3 - RESIDENTIAL USES DESCRIBED..txt
Wrote Text Successfully at /content/drive/MyDrive/corpus/§ 25-2-4 - COMMERCIAL USES DESCRIBED..txt
Wrote Text Successfully at /content/drive/MyDrive/corpus/§ 25-2-5 - INDUSTRIAL USES DESCRIBED..txt
Wrote Text Successfully at /content/drive/MyDrive/corpus/§ 25-2-6 - CIVIC USES DESCRIBED..txt
Wrote Text Successfully at /content/drive/MyDrive/corpus/§ 25-2-7 - AGRICULTURAL USES DESCRIBED..txt
Skipping empty doc ARTICLE 2. - ZONING DISTRICTS.


In [24]:
print("Total zoning nodes:", len(all_zoning_nodes))
print("Leaf nodes:", len(leaf_nodes))
print("Docs returned by test request:", len(docs))
print("Files in corpus:", len(os.listdir(corpus_dir)))

Total zoning nodes: 664
Leaf nodes: 566
Docs returned by test request: 92
Files in corpus: 8


In [25]:
test_indices = [0, 50, 100, 200, 300]
for i in test_indices:
    node = leaf_nodes[i]
    response = get_content(node["id"])
    print("\nREQUESTED:")
    print(node["heading"])
    print("DOCS RETURNED:", len(response["Docs"]))
    for doc in response["Docs"]:
        print(" ", doc["Title"])


Fetching children of: TIT25LADE_CH25-2ZO_SUBCHAPTER_AZOUSDIMADIDE_ART1ZOUS_S25-2-1USCL

REQUESTED:
§ 25-2-1 - USE CLASSIFICATIONS.
DOCS RETURNED: 92
  SUBCHAPTER A. - ZONING USES, DISTRICTS, AND MAP; DISTRICT DESIGNATIONS.
  ARTICLE 1. - ZONING USES.
  § 25-2-1 - USE CLASSIFICATIONS.
  § 25-2-2 - DETERMINATION OF USE CLASSIFICATION.
  § 25-2-3 - RESIDENTIAL USES DESCRIBED.
  § 25-2-4 - COMMERCIAL USES DESCRIBED.
  § 25-2-5 - INDUSTRIAL USES DESCRIBED.
  § 25-2-6 - CIVIC USES DESCRIBED.
  § 25-2-7 - AGRICULTURAL USES DESCRIBED.
  ARTICLE 2. - ZONING DISTRICTS.
  Division 1. - Districts Generally.
  § 25-2-31 - PURPOSE OF DISTRICTS.
  § 25-2-32 - ZONING DISTRICTS AND MAP CODES.
  § 25-2-33 - HIERARCHY OF BASE DISTRICTS.
  Division 2. - Residential Base Districts.
  § 25-2-51 - PURPOSES OF RESIDENTIAL DISTRICTS.
  § 25-2-52 - RESIDENTIAL DISTRICT DESIGNATIONS GENERALLY.
  § 25-2-53 - LAKE AUSTIN RESIDENCE (LA) DISTRICT DESIGNATION.
  § 25-2-54 - RURAL RESIDENCE (RR) DISTRICT DESIGNATION.


In [26]:
seen_ids = set()
all_docs = {}
request_count = 0

for node in leaf_nodes:
    if node["id"] in seen_ids: continue
    print(f"Requesting group containing: {node['heading']}")
    response = get_content(node["id"])
    request_count += 1
    for doc in response["Docs"]:
        seen_ids.add(doc["Id"])
        all_docs[doc["Id"]] = doc

print("API requests:", request_count)
print("Unique documents:", len(all_docs))

Requesting group containing: § 25-2-1 - USE CLASSIFICATIONS.
Fetching children of: TIT25LADE_CH25-2ZO_SUBCHAPTER_AZOUSDIMADIDE_ART1ZOUS_S25-2-1USCL
Requesting group containing: § 25-2-221 - DISTRICT DESIGNATION REQUIREMENTS.
Fetching children of: TIT25LADE_CH25-2ZO_SUBCHAPTER_BZOPRSPRECEDI_ART1ZOPRGE_DIV1DIDE_S25-2-221DIDERE
Requesting group containing: § 25-2-471 - INTERPRETATION GUIDELINES.
Fetching children of: TIT25LADE_CH25-2ZO_SUBCHAPTER_CUSDERE_ART1GEPR_S25-2-471INGU
Requesting group containing: § 25-2-491 - PERMITTED, CONDITIONAL, AND PROHIBITED USES.
Fetching children of: TIT25LADE_CH25-2ZO_SUBCHAPTER_CUSDERE_ART2PRUSDERE_DIV1RETA_S25-2-491PECOPRUS
Requesting group containing: § 25-2-551 - LAKE AUSTIN (LA) DISTRICT REGULATIONS.
Fetching children of: TIT25LADE_CH25-2ZO_SUBCHAPTER_CUSDERE_ART3ADRECEDI_DIV1REDI_S25-2-551LAAULADIRE
Requesting group containing: Division 1. - Residential Uses.
Fetching children of: TIT25LADE_CH25-2ZO_SUBCHAPTER_CUSDERE_ART4ADRECEUS_DIV1REUS
Requesti

In [27]:
leaf_ids = {node["id"] for node in leaf_nodes}
content_docs = {
    doc_id: doc for doc_id, doc in all_docs.items() if doc_id in leaf_ids
}

print("Leaf nodes expected:", len(leaf_nodes))
print("Leaf documents retrieved:", len(content_docs))

retrieved_ids = set(content_docs.keys())
missing_ids = leaf_ids - retrieved_ids
print(
    "Missing Nodes:",
    len(missing_ids)
)

Leaf nodes expected: 566
Leaf documents retrieved: 566
Missing Nodes: 0


In [28]:
node_lookup = {node["id"]: node for node in leaf_nodes}
corpus = []
for doc_id, doc in content_docs.items():
    node = node_lookup[doc_id]
    parsed = parse_section(doc)
    record = {
        "id": doc_id,
        "heading": node["heading"],
        "hierarchy": node["hierarchy"],
        "parent_id": node["parent_id"],
        "doc_order_id": node["doc_order_id"],
        "text": parsed["text"]
    }
    corpus.append(record)

In [29]:
print(json.dumps(corpus[0]))

{"id": "TIT25LADE_CH25-2ZO_SUBCHAPTER_AZOUSDIMADIDE_ART1ZOUS_S25-2-1USCL", "heading": "\u00a7 25-2-1 - USE CLASSIFICATIONS.", "hierarchy": ["CHAPTER 25-2. ZONING", "SUBCHAPTER A. - ZONING USES, DISTRICTS, AND MAP; DISTRICT DESIGNATIONS.", "ARTICLE 1. - ZONING USES.", "\u00a7 25-2-1 - USE CLASSIFICATIONS."], "parent_id": "TIT25LADE_CH25-2ZO_SUBCHAPTER_AZOUSDIMADIDE_ART1ZOUS", "doc_order_id": 225, "text": "This article describes and classifies uses in the zoning jurisdiction. The major use\n               categories are residential, commercial, industrial, civic, and agricultural.\nSource: Sections 13-2-2 through 13-2-6; Ord. 990225-70; Ord. 031211-11."}


In [30]:
import json
import os

jsonl_path = os.path.join(corpus_dir, "corpus.jsonl")

with open(jsonl_path, "w", encoding="utf-8") as f:
    for record in corpus:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

with open(jsonl_path, "r", encoding="utf-8") as f:
    for _ in range(3):
        print(json.loads(f.readline()))

{'id': 'TIT25LADE_CH25-2ZO_SUBCHAPTER_AZOUSDIMADIDE_ART1ZOUS_S25-2-1USCL', 'heading': '§ 25-2-1 - USE CLASSIFICATIONS.', 'hierarchy': ['CHAPTER 25-2. ZONING', 'SUBCHAPTER A. - ZONING USES, DISTRICTS, AND MAP; DISTRICT DESIGNATIONS.', 'ARTICLE 1. - ZONING USES.', '§ 25-2-1 - USE CLASSIFICATIONS.'], 'parent_id': 'TIT25LADE_CH25-2ZO_SUBCHAPTER_AZOUSDIMADIDE_ART1ZOUS', 'doc_order_id': 225, 'text': 'This article describes and classifies uses in the zoning jurisdiction. The major use\n               categories are residential, commercial, industrial, civic, and agricultural.\nSource: Sections 13-2-2 through 13-2-6; Ord. 990225-70; Ord. 031211-11.'}
{'id': 'TIT25LADE_CH25-2ZO_SUBCHAPTER_AZOUSDIMADIDE_ART1ZOUS_S25-2-2DEUSCL', 'heading': '§ 25-2-2 - DETERMINATION OF USE CLASSIFICATION.', 'hierarchy': ['CHAPTER 25-2. ZONING', 'SUBCHAPTER A. - ZONING USES, DISTRICTS, AND MAP; DISTRICT DESIGNATIONS.', 'ARTICLE 1. - ZONING USES.', '§ 25-2-2 - DETERMINATION OF USE CLASSIFICATION.'], 'parent_id': 'TI

In [31]:
lengths = [len(record["text"]) for record in corpus]

print("Documents:", len(corpus))
print("Shortest:", min(lengths))
print("Longest:", max(lengths))
print("Average:", sum(lengths) / len(lengths))

Documents: 566
Shortest: 0
Longest: 40868
Average: 1759.8727915194347


In [32]:
empty_sections = [record for record in corpus if not record["text"].strip()]
print("Empty sections:", len(empty_sections))

for record in corpus:
    if not record["text"].strip():
        print(record["heading"])
        print(record["id"])
        print()

corpus = [record for record in corpus if record["text"].strip()]

Empty sections: 11
Division 5. - Planned Unit Developments.
TIT25LADE_CH25-2ZO_SUBCHAPTER_BZOPRSPRECEDI_ART2SPRECEDI_DIV5PLUNDE

§ 25-2-514 - (RESERVED).
TIT25LADE_CH25-2ZO_SUBCHAPTER_CUSDERE_ART2PRUSDERE_DIV2REALDI_S25-2-514RE

§ 25-2-557 - RESERVED.
TIT25LADE_CH25-2ZO_SUBCHAPTER_CUSDERE_ART3ADRECEDI_DIV1REDI_S25-2-557RE

§ 25-2-564 - (RESERVED).
TIT25LADE_CH25-2ZO_SUBCHAPTER_CUSDERE_ART3ADRECEDI_DIV1REDI_S25-2-564RE

Division 2. - Commercial Districts.
TIT25LADE_CH25-2ZO_SUBCHAPTER_CUSDERE_ART3ADRECEDI_DIV2CODI

§ 25-2-646 - RESERVED.
TIT25LADE_CH25-2ZO_SUBCHAPTER_CUSDERE_ART3ADRECEDI_DIV5COOVDI_S25-2-646RE

Division 8. - Waterfront Overlay District and Subdistrict Development Regulations.
TIT25LADE_CH25-2ZO_SUBCHAPTER_CUSDERE_ART3ADRECEDI_DIV8WAOVDISUDERE

Division 10. - Transit Oriented Development District Regulations.
TIT25LADE_CH25-2ZO_SUBCHAPTER_CUSDERE_ART3ADRECEDI_DIV10TRORDEDIRE

Division 1. - Residential Uses.
TIT25LADE_CH25-2ZO_SUBCHAPTER_CUSDERE_ART4ADRECEUS_DIV1REUS

§ 2

In [45]:
def split_incr1(record, doc):
    soup = BeautifulSoup(doc["Content"], "html.parser")
    content = soup.find("div", class_="chunk-content")

    chunks = []
    current_parts = []
    current_marker = None
    chunk_number = 0

    for element in content.find_all(recursive=False):
        classes = element.get("class", [])
        # Don't include ordinance history in retrieval text
        if "historynote0" in classes: continue

        # Beginning of a new top-level definition
        if "incr1" in classes:

            # Save previous chunk
            if current_marker is not None:
                chunks.append({
                    "chunk_id": f"{record['id']}::definition::{chunk_number}",
                    "section_id": record["id"],
                    "section_heading": record["heading"],
                    "chunk_heading": current_marker,
                    "hierarchy": record["hierarchy"],
                    "text": "\n".join(current_parts)
                })

                chunk_number += 1

            current_marker = element.get_text(" ", strip=True)

            current_parts = [record["heading"], current_marker]

        # Once a definition has started, everything belongs to it
        # until the next incr1
        elif current_marker is not None:
            text = element.get_text("\n", strip=True)
            if text: current_parts.append(text)

    # Don't forget the final definition
    if current_marker is not None:
        chunks.append({
            "chunk_id": f"{record['id']}::definition::{chunk_number}",
            "section_id": record["id"],
            "section_heading": record["heading"],
            "chunk_heading": current_marker,
            "hierarchy": record["hierarchy"],
            "text": "\n".join(current_parts)
        })

    return chunks

In [35]:
def whole_section_chunk(record):
    return {
        "chunk_id": f"{record['id']}::0",
        "section_id": record["id"],
        "section_heading": record["heading"],
        "chunk_heading": record["heading"],
        "hierarchy": record["hierarchy"],
        "text": record["text"]
    }

In [43]:
def get_section_text(title):
    for doc in content_docs.values():
        if doc["Title"] == title:
            soup = BeautifulSoup(doc["Content"], "html.parser")
            return soup.get_text()

    return None

In [42]:
def get_section_code(title):
    for doc in content_docs.values():
        if doc["Title"] == title:
            soup = BeautifulSoup(doc["Content"], "html.parser")
            return soup.prettify()
    return None

In [47]:
CHUNK_THRESHOLD = 5000
chunked_corpus = []

for record in corpus:
    doc = content_docs[record["id"]]
    if len(record["text"]) <= CHUNK_THRESHOLD:
        chunked_corpus.extend(whole_section_chunk(record))
    else:
        subsection_chunks = split_incr1(record, doc)
        if subsection_chunks: chunked_corpus.append(subsection_chunks)
        else: print("Large doc needs another strategy: ", record["heading"])

Large doc needs another strategy:  § 2.4. - TIER TWO REQUIREMENTS.
Large doc needs another strategy:  § 2.5. - DEVELOPMENT BONUSES.
Large doc needs another strategy:  § 25-2-491 - PERMITTED, CONDITIONAL, AND PROHIBITED USES.
Large doc needs another strategy:  § 1.2. - APPLICABILITY.
Large doc needs another strategy:  § 1.5. - ALTERNATIVE EQUIVALENT COMPLIANCE.
Large doc needs another strategy:  § 2.2. - RELATIONSHIP OF BUILDINGS TO STREETS AND WALKWAYS.
Large doc needs another strategy:  § 2.3. - CONNECTIVITY BETWEEN SITES.
Large doc needs another strategy:  § 2.7. - PRIVATE COMMON OPEN SPACE AND PEDESTRIAN AMENITIES.
Large doc needs another strategy:  § 3.3. - OPTIONS TO IMPROVE BUILDING DESIGN.
Large doc needs another strategy:  § 4.2. - MIXED USE ZONING DISTRICTS.
Large doc needs another strategy:  § 4.3. - VERTICAL MIXED USE BUILDINGS.
Large doc needs another strategy:  ARTICLE 5: - DEFINITIONS.
Large doc needs another strategy:  APPENDIX B. - BOUNDARIES OF THE WATERFRONT OVERLAY D

In [44]:
print(get_section_code("§ 25-2-4 - COMMERCIAL USES DESCRIBED."))

<div class="chunk-content">
 <p class="incr0">
  (A)
 </p>
 <p class="content1">
  Commercial uses include the sale, rental, servicing, and distribution of goods, and
               the provision of services, other than those classified as industrial or civic uses.
 </p>
 <p class="incr0">
  (B)
 </p>
 <p class="content1">
  Commercial use classifications are described as follows:
 </p>
 <p class="incr1">
  (1)
 </p>
 <p class="content2">
  ADMINISTRATIVE AND BUSINESS OFFICES use is the use of a site for the provision of
               executive, management, or administrative services. This use includes:
 </p>
 <p class="incr2">
  (a)
 </p>
 <p class="content3">
  administrative offices and services, including real estate, insurance, property management,
               investment, personnel, travel, secretarial, telephone answering, and photocopy and
               reproduction; and
 </p>
 <p class="incr2">
  (b)
 </p>
 <p class="content3">
  business offices for public utilities, orga